## Summary

This notebook preprocesses the MIBiG 3.0 dataset for BGC classification:

### Data Processing Steps:
1. **Initial Loading**: Loaded MIBiG 3.0 data with 2,109 BGCs after duplicate removal
2. **Domain Mapping**: Converted Pfam IDs to domain names for BiGCARP compatibility
3. **Embedding Extraction**: Implemented extraction for three embedding approaches

### Embedding Methods:

**BiGCARP Embeddings:**
- Function ready for extracting embeddings from trained model checkpoints
- Supports different model variants (pretrained, random, frozen)
- Supports different layers (last, embedder)
- Code provided but commented out (requires model checkpoints)

**ESM Embeddings:**  
- Successfully extracted using pre-computed ESM embeddings of Pfam domains
- Applied max and mean pooling strategies
- Handles missing domains gracefully

**Pfam2vec Embeddings:**
- Successfully extracted using pre-computed pfam2vec embeddings
- Reports missing domains for transparency


The dataset is now ready for downstream BGC classification experiments!

In [1]:
import pandas as pd

# Load the initial MIBiG data obtained from the MIBiG website and deepbgc function
mibig_path = 'data/processed/bgc_product_classification/mibig_gbk_3.0_modified.tsv'
df_mibig = pd.read_csv(mibig_path, sep='\t')

df_pfam_sequences = df_mibig.groupby('sequence_id')['pfam_id'].apply(list).reset_index()
df_pfam_sequences.rename(columns={'pfam_id': 'pfam_sequence'}, inplace=True)
df_pfam_sequences.rename(columns={'sequence_id': 'bgc_id'}, inplace=True)

df_pfam_sequences.head()


,bgc_id,pfam_sequence
0,BGC0000001,"[PF02353, PF01135, PF01269, PF13489, PF01596, ..."
1,BGC0000002,"[PF00749, PF00201, PF04101, PF13579, PF03033, ..."
2,BGC0000003,"[PF00755, PF08659, PF00107, PF13489, PF10294, ..."
3,BGC0000004,"[PF07690, PF06609, PF00083, PF00975, PF00550, ..."
4,BGC0000006,"[PF07690, PF06609, PF00083, PF00975, PF12697, ..."


In [ ]:
# Load and process duplicates data
# data is from BiG-SCAPE
duplicates_path = "data/processed/bgc_product_classification/clustering.txt"
duplicates_df = pd.read_csv(duplicates_path, sep="\t")

# Clean up BGCs_id column and extract duplicates to drop
duplicates_df['BGCs_id'] = duplicates_df['BGCs_id'].str.lstrip(',')
duplicates_df['BGCs_to_drop'] = duplicates_df.apply(
    lambda row: ','.join(row['BGCs_id'].split(',')[1:]), axis=1
)

# Create list of duplicates to remove
duplicates_list = []
for bgcs_to_drop in duplicates_df['BGCs_to_drop']:
    if bgcs_to_drop:  # Skip empty strings
        duplicates_list.extend(bgcs_to_drop.split(','))

# Filter duplicates to only include those present in our data
current_bgc_ids = set(df_pfam_sequences['bgc_id'])
duplicates_to_remove = [bgc for bgc in duplicates_list if bgc in current_bgc_ids]

# Load classification labels
classes_path = 'data/processed/bgc_product_classification/MIBiG_3.0_classes.csv'
df_classes = pd.read_csv(classes_path)
Y = df_classes.set_index('sequence_id')

# Remove duplicates from pfam sequences data
df_processed = df_pfam_sequences[~df_pfam_sequences['bgc_id'].isin(duplicates_to_remove)].copy()

# Convert binary class columns to single product column
class_columns = ['Alkaloid', 'NRP', 'Other', 'Polyketide', 'RiPP', 'Saccharide', 'Terpene']

def get_product_string(row):
    """Convert binary class encoding to semicolon-separated product string"""
    products = []
    for col in class_columns:
        if row[col] == 1:
            products.append(col)
    return ';'.join(products) if products else 'Unknown'

# Merge with classification labels
df_final = df_processed.join(Y, on='bgc_id', how='left')
# Apply the conversion to create product column
df_final['product_class'] = df_final[class_columns].apply(get_product_string, axis=1)

# Drop the original binary class columns
df_final = df_final.drop(columns=class_columns)


print(f"\nShape: {df_final.shape}")
print(f"Removed {len(duplicates_to_remove)} duplicate BGCs")
df_final.head()




Shape: (2109, 3)
Removed 399 duplicate BGCs


,bgc_id,pfam_sequence,product_class
0,BGC0000001,"[PF02353, PF01135, PF01269, PF13489, PF01596, ...",Polyketide
1,BGC0000002,"[PF00749, PF00201, PF04101, PF13579, PF03033, ...",Polyketide
2,BGC0000003,"[PF00755, PF08659, PF00107, PF13489, PF10294, ...",Polyketide
3,BGC0000004,"[PF07690, PF06609, PF00083, PF00975, PF00550, ...",Polyketide
8,BGC0000010,"[PF07690, PF06609, PF00083, PF00106, PF08659, ...",Polyketide


## Summary

This notebook preprocesses the MIBiG 3.0 dataset for BGC classification:

### Data Processing Steps:
1. **Initial Loading**: Loaded MIBiG 3.0 data with 2,109 BGCs after duplicate removal
2. **Domain Mapping**: Converted Pfam IDs to domain names for BiGCARP compatibility
3. **Embedding Extraction**: Implemented extraction for three embedding approaches

### Embedding Methods:

**BiGCARP Embeddings:**
- Function ready for extracting embeddings from trained model checkpoints
- Supports different model variants (pretrained, random, frozen)
- Supports different layers (last, embedder)
- Code provided but commented out (requires model checkpoints)

**ESM Embeddings:**  
- Successfully extracted using pre-computed ESM embeddings of Pfam domains
- Applied max and mean pooling strategies
- Handles missing domains gracefully

**Pfam2vec Embeddings:**
- Successfully extracted using pre-computed pfam2vec embeddings
- Reports missing domains for transparency


The dataset is now ready for downstream BGC classification experiments!

In [ ]:
import json

# Convert Pfam sequences to domain sequences using domain mapping
mapping_path = 'data/processed/vocabularies/domain_to_pfam_mapping.json'
with open(mapping_path, 'r') as f:
    domain_to_pfam = json.load(f)

# Create reverse mapping: Pfam ID -> domain name
pfam_to_domain = {}
for domain, pfam_id in domain_to_pfam.items():
    # Remove version numbers from Pfam IDs
    pfam_base = pfam_id.split('.')[0] if '.' in pfam_id else pfam_id
    pfam_to_domain[pfam_base] = domain

print(f"Loaded {len(pfam_to_domain)} Pfam ID to domain name mappings")

# Function to convert a list of Pfam IDs to domain names
def convert_pfam_ids_to_domains(pfam_ids):
    domain_names = []
    for pfam_id in pfam_ids:
        # Try to match full ID first, then without version
        if pfam_id in pfam_to_domain:
            domain_names.append(pfam_to_domain[pfam_id])
        else:
            # Try without version number
            pfam_base = pfam_id.split('.')[0] if '.' in pfam_id else pfam_id
            if pfam_base in pfam_to_domain:
                domain_names.append(pfam_to_domain[pfam_base])
            else:
                # If no mapping is found, keep as UNK
                domain_names.append("UNK")
    return domain_names

# Apply the conversion to each sequence in the dataset
df_final['domain_sequence'] = df_final['pfam_sequence'].apply(convert_pfam_ids_to_domains)

# Check how many Pfams were successfully mapped
mapped_count = sum(1 for seq in df_final['domain_sequence'] for domain in seq if domain != "UNK")
total_count = sum(len(seq) for seq in df_final['domain_sequence'])
print(f"Successfully mapped {mapped_count} out of {total_count} Pfam IDs ({mapped_count/total_count*100:.2f}%)")

df_final.head()

Loaded 24076 Pfam ID to domain name mappings
Successfully mapped 102896 out of 103142 Pfam IDs (99.76%)


,bgc_id,pfam_sequence,product_class,domain_sequence
0,BGC0000001,"[PF02353, PF01135, PF01269, PF13489, PF01596, ...",Polyketide,"[CMAS, PCMT, Fibrillarin, Methyltransf_23, Met..."
1,BGC0000002,"[PF00749, PF00201, PF04101, PF13579, PF03033, ...",Polyketide,"[tRNA-synt_1c, UDPGT, Glyco_tran_28_C, Glyco_t..."
2,BGC0000003,"[PF00755, PF08659, PF00107, PF13489, PF10294, ...",Polyketide,"[Carn_acyltransf, KR, ADH_zinc_N, Methyltransf..."
3,BGC0000004,"[PF07690, PF06609, PF00083, PF00975, PF00550, ...",Polyketide,"[MFS_1, TRI12, Sugar_tr, Thioesterase, PP-bind..."
8,BGC0000010,"[PF07690, PF06609, PF00083, PF00106, PF08659, ...",Polyketide,"[MFS_1, TRI12, Sugar_tr, adh_short, KR, Epimer..."


In [ ]:
# Save the preprocessed dataset with domain sequences
import os
output_path = "data/processed/bgc_product_classification/processed_mibig3/mibig3_preprocessed.pkl"
df_final.to_pickle(output_path)
print(f"MIBiG 3.0 dataset with domain sequences saved to: {output_path}")

MIBiG 3.0 dataset with domain sequences saved to: data/processed/bgc_product_classification/processed_mibig3/mibig3_preprocessed.pkl


## Step 2: Extract Embeddings from Domain Sequences

Now we'll convert the domain sequences to sequences of embeddings using different approaches:
- BiGCARP embeddings (from different model/checkpoints/layers)
- ESM embeddings (using pre-computed ESM embeddings)
- Pfam2vec embeddings

### Required imports for embedding extraction

In [5]:
import torch
import json
from torch.utils.data import DataLoader, Dataset
from sequence_models.convolutional import ByteNetLM
from tqdm import tqdm
import numpy as np
from pathlib import Path
import sys
from cgrep import utils

/home/u5bb/han00.u5bb/miniforge3/envs/cgrep/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2.1 BiGCARP Embeddings

First, let's define the function to extract BiGCARP embeddings from trained checkpoints:

In [ ]:
import os
import json
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
from typing import Sequence

def extract_mibig3_embeddings(
    ckpt_path: str,
    bigcarp_vocab_path: str,
    mibig_data_path: str,
    output_path: str,
    layer_indices: Sequence[str] = ["last"],
    frozen: bool = False,
) -> None:
    """
    Extract layer embeddings from a single BIGCARP checkpoint for MIBiG 3.0 dataset.
    
    Parameters
    ----------
    ckpt_path : str
        Path to the specific BIGCARP checkpoint file.
    bigcarp_vocab_path : str
        JSON file with the BIGCARP vocabulary (must include "specials" and "domains").
    mibig_data_path : str
        Pickled pandas DataFrame containing MIBiG sequences under 'domain_sequence'.
    output_path : str
        Where to write the pickled DataFrame with embeddings.
    layer_indices : sequence[str], default ("last",)
        Which transformer layers to extract.
    frozen : bool, default False
        Whether to build a ByteNetLM with frozen embeddings.
    """
    
    # Load vocabulary
    with open(bigcarp_vocab_path, "r") as f:
        vocab_info = json.load(f)
    specials = vocab_info["specials"]
    domains = vocab_info["domains"]
    padding_idx = specials["-"]
    mask_idx = specials["#"]
    n_tokens = vocab_info["size"]

    # Load & tokenize MIBiG data
    mibig_data = pd.read_pickle(mibig_data_path)

    def tokenize_sequences(domain_sequences, domain_to_token):
        tokenized_sequences = []
        for seq in tqdm(domain_sequences, desc="Tokenizing sequences"):
            tokenized_sequences.append(
                [domain_to_token.get(domain, domain_to_token["UNK"]) for domain in seq]
            )
        return tokenized_sequences

    if "tokenized_sequence" not in mibig_data.columns:
        mibig_data["tokenized_sequence"] = tokenize_sequences(
            mibig_data["domain_sequence"], domains
        )

    # Prepare dataset/dataloader
    class PfamDataset(Dataset):
        def __init__(self, tokenized_sequences):
            self.tokenized_sequences = tokenized_sequences

        def __len__(self):
            return len(self.tokenized_sequences)

        def __getitem__(self, idx):
            return torch.tensor(self.tokenized_sequences[idx], dtype=torch.long)

    dataset = PfamDataset(mibig_data["tokenized_sequence"].tolist())
    dataloader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=lambda b: utils.mlm_collate_fn_extraction(
            b, mask_idx, padding_idx, mask_frac=0
        ),
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    # Build model (bc mode)
    model_kwargs = dict(
        n_tokens=n_tokens,
        d_embedding=1280,
        d_model=256,
        n_layers=32,
        kernel_size=3,
        r=128,
        slim=True,
        padding_idx=mask_idx,
        causal=False,
        final_ln=True,
        activation="gelu",
    )
    
    if frozen:
        model_kwargs["n_frozen_embs"] = len(domains) - 1

    model = ByteNetLM(**model_kwargs)
    
    # Load checkpoint
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
        
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval().to(device)

    # Extract embeddings
    all_embeddings = []
    for batch_tuple in tqdm(dataloader, desc="Extracting embeddings"):
        batch, mask = batch_tuple
        batch = batch.to(device)
        mask = mask.to(device)
        
        # Create attention mask (bc mode)
        input_mask = (batch != padding_idx).float().unsqueeze(-1)

        emb = utils.extract_layer_embeddings(
            model, batch, input_mask=input_mask, layer_indices=layer_indices
        )
        all_embeddings.extend(emb.detach().cpu().numpy())

    # Attach & save
    mibig_data["embeddings"] = all_embeddings
    mibig_data.to_pickle(output_path)
    
    # GPU housekeeping
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"Embeddings saved to: {output_path}")

#### Extract BiGCARP embeddings from different model checkpoints

Now let's run the embedding extraction for different BiGCARP models. **Note:** These will take significant time to run and require access to the trained model checkpoints.

In [7]:
# Extract BiGCARP embeddings from single checkpoints
# Extract ESM-pretrained initialization embeddings (last layer, checkpoint best)
extract_mibig3_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_esm_init/checkpoint_best.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="data/processed/bgc_product_classification/processed_mibig3/mibig3_preprocessed.pkl",
    output_path="artifacts/classification/mibig3/esm_init/mibig3_bigcarp_last.pkl",
    layer_indices=["last"],
    frozen=False,
)


Tokenizing sequences:   0%|          | 0/2109 [00:00<?, ?it/s]

Extracting embeddings: 100%|██████████| 2109/2109 [00:44<00:00, 47.62it/s]

Embeddings saved to: artifacts/classification/mibig3/esm_init/mibig3_bigcarp_last.pkl


In [8]:
extract_mibig3_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_esm_init/checkpoint_best.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="data/processed/bgc_product_classification/processed_mibig3/mibig3_preprocessed.pkl",
    output_path="artifacts/classification/mibig3/esm_init/mibig3_bigcarp_embedder.pkl",
    layer_indices=["embedder"],
    frozen=False,
)

Tokenizing sequences: 100%|██████████| 2109/2109 [00:00<00:00, 181251.27it/s]


Extracting embeddings: 100%|██████████| 2109/2109 [00:00<00:00, 3465.02it/s]


Embeddings saved to: artifacts/classification/mibig3/esm_init/mibig3_bigcarp_embedder.pkl


In [9]:
# Random initialization embeddings (last layer)
extract_mibig3_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_random_init/checkpoint_best.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="data/processed/bgc_product_classification/processed_mibig3/mibig3_preprocessed.pkl",
    output_path="artifacts/classification/mibig3/random_init/mibig3_bigcarp_last.pkl",
    layer_indices=["last"],
    frozen=False,
)


Tokenizing sequences: 100%|██████████| 2109/2109 [00:00<00:00, 201585.81it/s]


Extracting embeddings: 100%|██████████| 2109/2109 [00:43<00:00, 48.16it/s]

Embeddings saved to: artifacts/classification/mibig3/random_init/mibig3_bigcarp_last.pkl


In [ ]:
# Random initialization embeddings (embedder)
extract_mibig3_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_random_init/checkpoint_best.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="data/processed/bgc_product_classification/processed_mibig3/mibig3_preprocessed.pkl",
    output_path="artifacts/classification/mibig3/random_init/mibig3_bigcarp_embedder.pkl",
    layer_indices=["embedder"],
    frozen=False,
)

Tokenizing sequences: 100%|██████████| 2109/2109 [00:00<00:00, 196289.52it/s]


Extracting embeddings: 100%|██████████| 2109/2109 [00:00<00:00, 3502.75it/s]


Embeddings saved to: artifacts/classification/mibig3/random_init/mibig3_bigcarp_embedder.pkl


In [7]:
# Random initialization embeddings (embedder) ckp00
extract_mibig3_embeddings(
    ckpt_path="artifacts/bigcarp/bigcarp_models/run_random_init/checkpoint00.tar",
    bigcarp_vocab_path="data/processed/vocabularies/pfam_vocab_present.json",
    mibig_data_path="data/processed/bgc_product_classification/processed_mibig3/mibig3_preprocessed.pkl",
    output_path="artifacts/classification/mibig3/random_init/mibig3_bigcarp_embedder_00.pkl",
    layer_indices=["embedder"],
    frozen=False,
)

Tokenizing sequences:   0%|          | 0/2109 [00:00<?, ?it/s]

Extracting embeddings: 100%|██████████| 2109/2109 [00:00<00:00, 2984.43it/s]


Embeddings saved to: artifacts/classification/mibig3/random_init/mibig3_bigcarp_embedder_00.pkl


### 2.2 ESM Embeddings

Extract ESM embeddings using pre-computed ESM embeddings of Pfam domains:

In [11]:
# Load the preprocessed MIBiG 3.0 data
mibig3_data = pd.read_pickle("data/processed/bgc_product_classification/processed_mibig3/mibig3_preprocessed.pkl")

# Load existing ESM embeddings of pfam domains
emb_matrix = torch.load("artifacts/bigcarp/esm_embeddings/esm1b_pfam_embs.pt", weights_only=False)

# Load the vocab of pfam domains which encodes the domains' positions in the embedding matrix
with open("data/processed/vocabularies/pfam_vocab.json", 'r') as f:
    vocab = json.load(f)
domains = vocab["domains"]

# Remove the UNK domain and create mapping from domain to index
domains = {k: v for k, v in domains.items() if k != "UNK"}
domain_to_idx = {domain: idx for idx, domain in enumerate(domains.keys())}

def get_domain_embeddings_esm(domain_sequence, domain_to_idx, emb_matrix):
    """
    Given a sequence of domain names, return the corresponding embeddings from ESM's embeddings matrix.
    """
    # Get the indices for the domains in the sequence
    indices = [domain_to_idx[domain] for domain in domain_sequence if domain in domain_to_idx]
    # Get the corresponding embeddings
    if len(indices) > 0:
        embeddings = emb_matrix[indices]
    else:
        # Return empty tensor if no domains found
        embeddings = torch.empty(0, emb_matrix.shape[1])
    return embeddings

# Apply the function to the dataset
print("Extracting ESM embeddings for domain sequences...")
mibig3_data['esm_embeddings'] = mibig3_data['domain_sequence'].apply(
    lambda seq: get_domain_embeddings_esm(seq, domain_to_idx, emb_matrix)
)

# Check for empty embeddings
empty_embeddings = mibig3_data['esm_embeddings'].apply(lambda emb: emb.shape[0] == 0)
print(f"Number of entries with empty embeddings: {empty_embeddings.sum()}")

# Display entries with empty embeddings
if empty_embeddings.any():
    print("Entries with empty embeddings (first 5):")
    print(mibig3_data[empty_embeddings].head()[['bgc_id', 'domain_sequence']])

Extracting ESM embeddings for domain sequences...
Number of entries with empty embeddings: 0


In [12]:
# Save the dataset with ESM embeddings
esm_output_path = "artifacts/classification/mibig3/mibig3_esm_embeddings.pkl"

# Convert ESM embeddings from tensors to numpy arrays for easier serialization
print("Converting ESM embeddings from tensors to numpy arrays...")
mibig3_data['esm_embeddings'] = mibig3_data['esm_embeddings'].apply(
    lambda emb: emb.detach().cpu().numpy() if isinstance(emb, torch.Tensor) else emb
)

mibig3_data.to_pickle(esm_output_path)
print(f"MIBiG 3.0 dataset with ESM embeddings saved to: {esm_output_path}")


mibig3_data.head()

Converting ESM embeddings from tensors to numpy arrays...
MIBiG 3.0 dataset with ESM embeddings saved to: artifacts/classification/mibig3/mibig3_esm_embeddings.pkl


,bgc_id,pfam_sequence,product_class,domain_sequence,esm_embeddings
0,BGC0000001,"[PF02353, PF01135, PF01269, PF13489, PF01596, ...",Polyketide,"[CMAS, PCMT, Fibrillarin, Methyltransf_23, Met...","[[-0.031329084, 0.11174197, -0.22890997, 0.016..."
1,BGC0000002,"[PF00749, PF00201, PF04101, PF13579, PF03033, ...",Polyketide,"[tRNA-synt_1c, UDPGT, Glyco_tran_28_C, Glyco_t...","[[0.017248986, -0.012069668, -0.23262857, 0.05..."
2,BGC0000003,"[PF00755, PF08659, PF00107, PF13489, PF10294, ...",Polyketide,"[Carn_acyltransf, KR, ADH_zinc_N, Methyltransf...","[[-0.0030203925, 0.23956646, 0.21761517, 0.129..."
3,BGC0000004,"[PF07690, PF06609, PF00083, PF00975, PF00550, ...",Polyketide,"[MFS_1, TRI12, Sugar_tr, Thioesterase, PP-bind...","[[-0.010534718, 0.12279355, -0.05196583, -0.16..."
8,BGC0000010,"[PF07690, PF06609, PF00083, PF00106, PF08659, ...",Polyketide,"[MFS_1, TRI12, Sugar_tr, adh_short, KR, Epimer...","[[-0.010534718, 0.12279355, -0.05196583, -0.16..."


### 2.3 Pfam2vec Embeddings

Extract Pfam2vec embeddings using the pre-computed pfam2vec embeddings:

In [13]:
# Load MIBiG 3.0 data (without ESM embeddings to save memory for pfam2vec processing)
mibig3_data_pfam = mibig3_data.drop(columns=['esm_embeddings', 'esm_max_embedding', 'esm_mean_embedding'], errors='ignore')

# Load pfam2vec embeddings
pfam2vec_path = 'data/processed/bgc_product_classification/pfam2vec.csv'
pfam2vec_df = pd.read_csv(pfam2vec_path)

print(f"Loaded pfam2vec embeddings with shape: {pfam2vec_df.shape}")
print(f"Pfam2vec embedding dimension: {pfam2vec_df.shape[1] - 1}")  # -1 for pfam_id column

# Function to convert PFAM sequence to sequence of embeddings
def get_pfam_sequence_embeddings_and_missing(pfam_sequence, pfam_df):
    """
    Convert a sequence of PFAM domains to their corresponding embeddings.
    
    Args:
        pfam_sequence: List of PFAM domains
        pfam_df: DataFrame containing PFAM domain embeddings with 'pfam_id' column
        
    Returns:
        Tuple: (List of embedding vectors, List of missing PFAM IDs)
    """
    if isinstance(pfam_sequence, list):
        domains = pfam_sequence
    else:
        return [], []  # Return empty if not a list
    
    embeddings = []
    missing_domains_in_sequence = []
    
    for domain in domains:
        if 'pfam_id' not in pfam_df.columns:
            print("Error: 'pfam_id' column not found in pfam_df")
            return [], domains
            
        domain_match = pfam_df[pfam_df['pfam_id'] == domain]
        if not domain_match.empty:
            # Get embedding vector (all columns except pfam_id)
            embedding = domain_match.iloc[0, pfam_df.columns != 'pfam_id'].values.astype(np.float32)
            embeddings.append(embedding)
        else:
            missing_domains_in_sequence.append(domain)
    
    return embeddings, missing_domains_in_sequence

print("Extracting pfam2vec embeddings for PFAM sequences...")

# Apply the function to create pfam2vec embeddings
results_tuples = mibig3_data_pfam['pfam_sequence'].apply(
    lambda x: get_pfam_sequence_embeddings_and_missing(x, pfam2vec_df)
)

# Separate embeddings and missing domains
mibig3_data_pfam['pfam2vec_seq'] = results_tuples.str[0]
list_of_missing_pfams_per_bgc = results_tuples.str[1]

# Summarize missing PFAM IDs
overall_unique_missing_pfams = set()
for missing_list in list_of_missing_pfams_per_bgc:
    if missing_list:
        overall_unique_missing_pfams.update(missing_list)

if overall_unique_missing_pfams:
    print(f"\nTotal of {len(overall_unique_missing_pfams)} unique PFAM IDs not found in pfam2vec embeddings:")
    if len(overall_unique_missing_pfams) <= 20:  # Show if not too many
        for pfam_id in sorted(list(overall_unique_missing_pfams)):
            print(f"  - {pfam_id}")
    else:
        print(f"  (showing first 20): {sorted(list(overall_unique_missing_pfams))[:20]}")
    print("Missing domains were skipped during embedding generation.")
else:
    print("All PFAM IDs were successfully found in the pfam2vec embeddings.")

# Check for entries with empty pfam2vec embeddings
empty_pfam2vec = mibig3_data_pfam['pfam2vec_seq'].apply(lambda x: len(x) == 0)
print(f"Number of entries with empty pfam2vec embeddings: {empty_pfam2vec.sum()}")

print(f"Pfam2vec processing completed. Dataset shape: {mibig3_data_pfam.shape}")
mibig3_data_pfam.head()

Loaded pfam2vec embeddings with shape: (13312, 101)
Pfam2vec embedding dimension: 100
Extracting pfam2vec embeddings for PFAM sequences...

Total of 109 unique PFAM IDs not found in pfam2vec embeddings:
  (showing first 20): ['PF00098', 'PF00172', 'PF00241', 'PF00262', 'PF00319', 'PF00399', 'PF00693', 'PF00780', 'PF00789', 'PF00827', 'PF00974', 'PF01088', 'PF01185', 'PF01328', 'PF01480', 'PF02072', 'PF02184', 'PF02197', 'PF02269', 'PF02721']
Missing domains were skipped during embedding generation.
Number of entries with empty pfam2vec embeddings: 0
Pfam2vec processing completed. Dataset shape: (2109, 5)


,bgc_id,pfam_sequence,product_class,domain_sequence,pfam2vec_seq
0,BGC0000001,"[PF02353, PF01135, PF01269, PF13489, PF01596, ...",Polyketide,"[CMAS, PCMT, Fibrillarin, Methyltransf_23, Met...","[[0.09173657, 0.035680633, 0.0135731, -0.12299..."
1,BGC0000002,"[PF00749, PF00201, PF04101, PF13579, PF03033, ...",Polyketide,"[tRNA-synt_1c, UDPGT, Glyco_tran_28_C, Glyco_t...","[[0.14181772, -0.007126024, 0.007510652, -0.07..."
2,BGC0000003,"[PF00755, PF08659, PF00107, PF13489, PF10294, ...",Polyketide,"[Carn_acyltransf, KR, ADH_zinc_N, Methyltransf...","[[-0.018139565, -0.08186026, -0.14477561, -0.0..."
3,BGC0000004,"[PF07690, PF06609, PF00083, PF00975, PF00550, ...",Polyketide,"[MFS_1, TRI12, Sugar_tr, Thioesterase, PP-bind...","[[-0.03312874, 0.0647246, -0.062299524, -0.002..."
8,BGC0000010,"[PF07690, PF06609, PF00083, PF00106, PF08659, ...",Polyketide,"[MFS_1, TRI12, Sugar_tr, adh_short, KR, Epimer...","[[-0.03312874, 0.0647246, -0.062299524, -0.002..."


In [14]:
# Save the dataset with pfam2vec embeddings
pfam2vec_output_path = "artifacts/classification/mibig3/mibig3_pfam2vec_embeddings.pkl"
mibig3_data_pfam.to_pickle(pfam2vec_output_path)
print(f"MIBiG 3.0 dataset with pfam2vec embeddings saved to: {pfam2vec_output_path}")

mibig3_data_pfam.head()


MIBiG 3.0 dataset with pfam2vec embeddings saved to: artifacts/classification/mibig3/mibig3_pfam2vec_embeddings.pkl


,bgc_id,pfam_sequence,product_class,domain_sequence,pfam2vec_seq
0,BGC0000001,"[PF02353, PF01135, PF01269, PF13489, PF01596, ...",Polyketide,"[CMAS, PCMT, Fibrillarin, Methyltransf_23, Met...","[[0.09173657, 0.035680633, 0.0135731, -0.12299..."
1,BGC0000002,"[PF00749, PF00201, PF04101, PF13579, PF03033, ...",Polyketide,"[tRNA-synt_1c, UDPGT, Glyco_tran_28_C, Glyco_t...","[[0.14181772, -0.007126024, 0.007510652, -0.07..."
2,BGC0000003,"[PF00755, PF08659, PF00107, PF13489, PF10294, ...",Polyketide,"[Carn_acyltransf, KR, ADH_zinc_N, Methyltransf...","[[-0.018139565, -0.08186026, -0.14477561, -0.0..."
3,BGC0000004,"[PF07690, PF06609, PF00083, PF00975, PF00550, ...",Polyketide,"[MFS_1, TRI12, Sugar_tr, Thioesterase, PP-bind...","[[-0.03312874, 0.0647246, -0.062299524, -0.002..."
8,BGC0000010,"[PF07690, PF06609, PF00083, PF00106, PF08659, ...",Polyketide,"[MFS_1, TRI12, Sugar_tr, adh_short, KR, Epimer...","[[-0.03312874, 0.0647246, -0.062299524, -0.002..."
